# IBF Domain I: Rotating Rules World (RRW)

This notebook runs the **Stage I mechanistic validation** of the IBF framework (§6.2, §7.1). RRW is a controlled non-stationary environment with analytically known structure, used to test the mechanism under explicit cross-context contradiction.

## What This Notebook Tests

Three predictions from the experimental program (§6.2):

| Prediction | Test | Key Metric |
|---|---|---|
| **P1** — Retention Without Replay | IBF achieves BT competitive with or superior to MLP baselines without replaying raw data | BT_A = −0.234 vs Replay −0.410 (43% reduction) |
| **P2** — Crucible-Mediated Transfer | No-Crucible achieves BT_A ≈ 0 (perfect isolation); the Crucible enables transfer at modest retention cost | No-Crucible BT_A = −0.005 |
| **P3** — Forward Learning Without Overwrite | IBF achieves final-phase accuracy exceeding baselines | Acc_C = 0.913 vs Replay 0.816 |

## Environment Structure (§5.2)

RRW operates in an **8-dimensional latent space** `z = [x_4D; action_emb_4D]` across three sequential phases (A → B → C). The score decomposes into:
- An **invariant** component (x₁, always valid)
- A **shared** phase-weighted component (x₂)
- A **contextual** 2D component (x₃₋₄) that **reverses** between Phase A and B

Phase B is the exact reversal of Phase A on the contextual subspace. Phase C introduces a new random orientation.

## Notebook Structure

| Cell | Role | Paper Section |
|------|------|---------------|
| 1 | Setup & dependencies | — |
| 2 | Configuration (all v4.17 parameters) | §5.2 |
| 3 | RRW environment (3-phase scoring) | §5.2, Eq. 1 analog |
| 4 | Encoder, base evaluator, σ calibration | §5.1 |
| 5 | κ measurement (geometric scaling ratio) | §5.1, Table in §5.1 |
| 6 | **IBF Engine** (Read Path, Write Path, Lifecycle) | §4 |
| 7 | Baselines (MLP, Replay MLP, Passive) | §5.2 |
| 8 | Full IBF training (5 seeds) + baselines | §7.1 |
| 8b/c/d | Ablations: No-Agency, No-Cryst, No-Crucible (5 seeds each) | §7.1, Table 1 |
| 9 | Evaluation, Table 1 reproduction, reproduction check | §7.1 |
| 10 | σ_eval sweep, κ report, results export | §5.1, §7.1 |
| 11 | Geometric interference prediction (exploratory, not in paper) | — |

## Reproducibility

- **5 seeds**: [42, 43, 44, 45, 46]
- All seeds set via `np.random.seed()` and `torch.manual_seed()`
- Baselines trained in the same loop on identical data
- Engine states saved after each phase for post-hoc analysis
- Cell 9 includes an automated reproduction check against expected values

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — SETUP & DEPENDENCIES                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Rotating Rules World (RRW) — IBF Paper-Grade Experiment
# Clean restructuring of v4.17 for cross-domain consistency.
# Same Cell 21B logic. Same parameters. Same seeds.

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from collections import defaultdict
import json, sys, time, os, pickle, copy

try:
    import matplotlib
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "matplotlib"])
    import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print("RRW Paper-Grade Notebook — v4.17 Clean")
print(f"  NumPy {np.__version__}, PyTorch {torch.__version__}")

RRW Paper-Grade Notebook — v4.17 Clean
  NumPy 2.1.2, PyTorch 2.8.0+cu128


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — CONFIGURATION                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# All parameters from v4.17 — DO NOT CHANGE.

SEEDS = [42, 43, 44, 45, 46]
OUTPUT_DIR = "rrw_paper_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

@dataclass
class Config:
    # ── Environment ──
    d: int = 4; k: int = 4
    beta: float = 1.0; alpha: float = 1.0
    gamma_A: float = 0.8; gamma_B: float = 0.5; gamma_C: float = 0.8
    sigma_eps: float = 0.25
    N_pool: int = 1000; E: int = 25; N_test: int = 2000
    eval_every: int = 100
    # ── IBF core ──
    eta: float = 0.10; eta_cryst: float = 0.005
    mu_base: float = 0.06; mu_cryst: float = 0.001
    v_max: float = 0.50
    # ── Crystallization ──
    cryst_n_min: int = 15
    activation_thresh: float = 0.18
    creation_thresh: float = 0.30
    convergence_threshold: float = 0.075  # NOTE: differs from chess (0.025)
    capacity: int = 2000
    # ── Thermodynamic shrink ──
    alpha_shrink: float = 1.0  # NOTE: shrink disabled in RRW
    sigma_floor: float = 0.15
    min_samples_shrink: int = 50
    # ── Agency channel ──
    k_0: float = 5.0; k_min: float = 1.0
    eta_k: float = 0.05
    w_max: float = 5.0
    w_dvar_threshold: float = 0.045  # NOTE: differs from chess (0.005)
    # ── Crucible & dissolution ──
    n_cross_min: int = 8
    reversal_threshold: float = -0.125  # NOTE: differs from chess (-0.0125)
    # ── MLP baseline ──
    mlp_lr: float = 0.01
    # ── Action embedding calibration ──
    action_sep_target: float = 3.5; action_sep_margin: float = 4.0
    action_sep_max_iter: int = 5

C = Config()

p_vec = np.array([+1., -1., +1., -1.])
q_vec = np.array([+1., +1., -1., -1.])
r_vec = np.array([+1., -1., -1., +1.])
ACTION_EMB = np.eye(C.k)

SIGMA_EVAL_SCALES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.5]

print(f"  Seeds: {SEEDS}")
print(f"  η={C.eta}, η_cryst={C.eta_cryst}, μ_base={C.mu_base}, μ_cryst={C.mu_cryst}")
print(f"  η/μ transient={C.eta/C.mu_base:.2f}, crystal={C.eta_cryst/C.mu_cryst:.1f}")

  Seeds: [42, 43, 44, 45, 46]
  η=0.1, η_cryst=0.005, μ_base=0.06, μ_cryst=0.001
  η/μ transient=1.67, crystal=5.0


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — RRW ENVIRONMENT                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

class Environment:
    def __init__(self, seed):
        rng = np.random.RandomState(seed)
        self.rng_seed = seed
        self.u_A = rng.randn(2); self.u_A /= np.linalg.norm(self.u_A)
        self.u_B = -self.u_A
        self.u_C = rng.randn(2); self.u_C /= np.linalg.norm(self.u_C)
        self.u = {'A': self.u_A, 'B': self.u_B, 'C': self.u_C}
        self.gamma = {'A': C.gamma_A, 'B': C.gamma_B, 'C': C.gamma_C}
        self.pool = rng.randn(C.N_pool, C.d)
        self.pools = {'A': self.pool, 'B': self.pool, 'C': self.pool}
        self.tests = {ctx: rng.randn(C.N_test, C.d) for ctx in 'ABC'}
        noise_rng = np.random.RandomState(seed + 100)
        self.pool_noise = {
            ctx: noise_rng.randn(C.E, C.N_pool, C.k) * C.sigma_eps
            for ctx in 'ABC'
        }

    def score_clean(self, x, ctx):
        u_c, g_c = self.u[ctx], self.gamma[ctx]
        s = np.zeros(C.k)
        for j in range(C.k):
            s[j] = (C.beta * x[0] * p_vec[j]
                     + g_c * x[1] * q_vec[j]
                     + C.alpha * np.dot(u_c, x[2:4]) * r_vec[j])
        return s

    def correct_action_noisy(self, x, ctx, epoch, pool_idx):
        return int(np.argmax(
            self.score_clean(x, ctx) + self.pool_noise[ctx][epoch, pool_idx]))

    def correct_action_clean(self, x, ctx):
        return int(np.argmax(self.score_clean(x, ctx)))

    def correct_actions_batch(self, X, ctx):
        u_c, g_c = self.u[ctx], self.gamma[ctx]
        N = len(X); S = np.zeros((N, C.k))
        for j in range(C.k):
            S[:, j] = (C.beta * X[:, 0] * p_vec[j]
                        + g_c * X[:, 1] * q_vec[j]
                        + C.alpha * (X[:, 2:4] @ u_c) * r_vec[j])
        return np.argmax(S, axis=1)

print("Environment ready.")

Environment ready.


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — ENCODER (RawEncoder — frozen passthrough)             ║
# ╚══════════════════════════════════════════════════════════════════╝

class RawEncoder:
    def __init__(self, seed=None): pass
    def encode(self, x_np, action_idx):
        return np.concatenate([x_np, ACTION_EMB[action_idx]])
    def encode_batch(self, x_np, action_indices):
        return np.concatenate([x_np, ACTION_EMB[action_indices]], axis=1)

class BaseEvaluator:
    def __init__(self, seed, encoder=None):
        rng = np.random.RandomState(seed)
        z_dim = C.d + C.k
        variances = [0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
        self.w = rng.randn(z_dim) * 0.01; self.b = 0.0
        if encoder is None: return
        for var in variances:
            self.w = rng.randn(z_dim) * var; self.b = 0.0
            spread = self._check_spread(encoder, rng)
            if spread >= 0.02:
                print(f"  Base evaluator spread: {spread:.4f} (init variance={var})"); return
        print(f"  Base evaluator spread: {spread:.4f} (max variance, below target)")

    def _check_spread(self, encoder, rng):
        xs = rng.randn(250, C.d); acts = rng.randint(0, C.k, size=250)
        Z = encoder.encode_batch(xs, acts); spreads = []
        for i in range(0, 250 - C.k + 1, C.k):
            vals = self.batch(Z[i:i+C.k]); spreads.append(vals.max() - vals.min())
        return float(np.mean(spreads))

    def __call__(self, z):
        return 1.0 / (1.0 + np.exp(-(np.dot(self.w, z) + self.b)))
    def batch(self, Z):
        return 1.0 / (1.0 + np.exp(-(Z @ self.w + self.b)))

def _quick_sigma_calibration(encoder, seed):
    rng = np.random.RandomState(seed + 999)
    xs = rng.randn(500, C.d)
    Z = encoder.encode_batch(xs, np.zeros(500, dtype=int))
    Z_c = Z - Z.mean(axis=0); _, s, _ = np.linalg.svd(Z_c, full_matrices=False)
    var_exp = np.cumsum(s**2) / np.sum(s**2)
    eff_rank = int(np.searchsorted(var_exp, 0.95)) + 1
    dists = []
    for i in range(min(500, len(Z))):
        for j in range(i+1, min(i+20, len(Z))):
            dists.append(np.linalg.norm(Z[i] - Z[j]))
    med_dist = np.median(dists)
    return med_dist / np.sqrt(2 * max(eff_rank, 1)), eff_rank, med_dist

def measure_action_separation(encoder, sigma, seed):
    rng = np.random.RandomState(seed + 500); xs = rng.randn(200, C.d); seps = []
    for x in xs:
        zs = [encoder.encode(x, j) for j in range(C.k)]
        for a in range(C.k):
            for b in range(a+1, C.k): seps.append(np.linalg.norm(zs[a] - zs[b]))
    m = np.mean(seps); return m, m / sigma if sigma > 0 else 0.0

def calibrate_action_embedding(encoder, seed, log_print=print):
    global ACTION_EMB
    ACTION_EMB = np.eye(C.k); emb_scale = 1.0
    sigma_est, _, _ = _quick_sigma_calibration(encoder, seed)
    for attempt in range(C.action_sep_max_iter):
        mean_sep, ratio = measure_action_separation(encoder, sigma_est, seed)
        log_print(f"  [Sep iter {attempt+1}] sep={mean_sep:.4f}, σ={sigma_est:.4f}, ratio={ratio:.2f}")
        if ratio >= C.action_sep_target:
            log_print(f"  Action separation OK (ratio={ratio:.2f})"); break
        t = C.action_sep_margin * sigma_est / mean_sep
        emb_scale *= t; ACTION_EMB = np.eye(C.k) * emb_scale
        log_print(f"  Rescaled ACTION_EMB {t:.2f}x (cumulative: {emb_scale:.2f}x)")
        sigma_est, _, _ = _quick_sigma_calibration(encoder, seed)
    return sigma_est, emb_scale

def run_diagnostics(encoder, base_eval):
    rng = np.random.RandomState(42 + 999); xs = rng.randn(500, C.d)
    Z = encoder.encode_batch(xs, np.zeros(500, dtype=int))
    Z_c = Z - Z.mean(axis=0); _, s, _ = np.linalg.svd(Z_c, full_matrices=False)
    var_exp = np.cumsum(s**2) / np.sum(s**2)
    eff_rank = int(np.searchsorted(var_exp, 0.95)) + 1
    dists = []
    for i in range(min(500, len(Z))):
        for j in range(i+1, min(i+20, len(Z))):
            dists.append(np.linalg.norm(Z[i] - Z[j]))
    sigma_cal = np.median(dists) / np.sqrt(2 * max(eff_rank, 1))
    merge_thresh = 1.0 * sigma_cal
    print(f"  σ={sigma_cal:.4f}, merge={merge_thresh:.4f}, eff_rank={eff_rank}")
    return eff_rank, sigma_cal, merge_thresh

print("Encoder & calibration ready.")

Encoder & calibration ready.


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — σ CALIBRATION & κ MEASUREMENT                        ║
# ╚══════════════════════════════════════════════════════════════════╝

def compute_kappa(encoder, seed, log_print=print):
    rng = np.random.RandomState(seed + 999)
    xs = rng.randn(1000, C.d)
    cal_z_8d = encoder.encode_batch(xs, np.zeros(1000, dtype=int))
    eigenvalues = np.var(cal_z_8d, axis=0)
    eigenvalues = np.sort(eigenvalues)[::-1]
    d_eff = (np.sum(eigenvalues)**2) / np.sum(eigenvalues**2)
    sigma_cal, _, _ = _quick_sigma_calibration(encoder, seed)
    kappa = sigma_cal / np.sqrt(d_eff)
    log_print(f"\n  ╔══════════════════════════════════════════╗")
    log_print(f"  ║  d_eff (participation ratio) = {d_eff:.2f}       ║")
    log_print(f"  ║  σ_rrw = {sigma_cal:.4f}                       ║")
    log_print(f"  ║  κ_rrw = {sigma_cal:.4f} / √{d_eff:.1f} = {kappa:.4f}  ║")
    log_print(f"  ╚══════════════════════════════════════════╝")
    return {'d_eff': float(d_eff), 'sigma': float(sigma_cal),
            'kappa': float(kappa), 'eigenvalues': eigenvalues.tolist()}

print("κ measurement ready.")

κ measurement ready.


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — IBF ENGINE                                             ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# This is the RRW-native implementation of IBF engine. It matches all
# other domain implementations but uses the RRW-specific encoding (8D z =
# [x_4d; action_emb_4d]). Logic is UNCHANGED

@dataclass
class MemoryCenter:
    z: np.ndarray
    v: float = 0.0
    w: float = 0.0
    n_updates: int = 0
    D_sum: float = 0.0
    D_sq_sum: float = 0.0
    mu_eff: float = 0.06
    context_id: int = -1
    birth_step: int = 0
    context_update_counts: Dict[int, int] = field(default_factory=lambda: defaultdict(int))
    sigma: float = 0.58
    D_history: List[float] = field(default_factory=list)
    D_step_history: List[int] = field(default_factory=list)
    xj_history: List[Tuple] = field(default_factory=list)
    was_ever_crystallized: bool = False
    crucible_verified: bool = False
    dissolution_log: List[Dict] = field(default_factory=list)

    def D_var(self):
        """Rolling variance: skip first 20 (birth trauma), use last 50."""
        if self.n_updates < 20:
            return 0.0
        rec = self.D_history[20:][-50:]
        if len(rec) < 2:
            return 0.0
        return float(np.var(rec))

    def is_crystallized(self):
        return self.mu_eff < C.mu_base - 1e-6

    def n_cross_updates(self):
        return sum(cnt for ctx, cnt in self.context_update_counts.items()
                   if ctx != self.context_id)


class IBFAgent:
    def __init__(self, sigma, merge_threshold, encoder, base_eval,
                 enable_crystallization=True, enable_crucible=True,
                 enable_agency=True):
        self.centers: List[MemoryCenter] = []
        self.sigma_base = sigma
        self.merge_thresh = merge_threshold
        self.encoder = encoder
        self.base_eval = base_eval
        self.current_context = -1
        self.global_step = 0
        self.enable_crystallization = enable_crystallization
        self.enable_crucible = enable_crucible
        self.enable_agency = enable_agency
        self._merge_ratio = merge_threshold / sigma
        self._dynamic_sigma_floor = min(C.sigma_floor, sigma * 0.25)
        self._needle_threshold = sigma * 0.50

    def set_context(self, ctx_id):
        if ctx_id != self.current_context:
            for c in self.centers:
                c.crucible_verified = False
        self.current_context = ctx_id

    def kernel_batch(self, z, centers=None):
        cs = centers if centers is not None else self.centers
        if not cs:
            return np.array([])
        Z = np.array([c.z for c in cs])
        sq = np.sum((Z - z[np.newaxis, :])**2, axis=1)
        sigmas = np.array([c.sigma for c in cs])
        return np.exp(-sq / (2.0 * sigmas**2))

    def _read_gate(self, c):
        if c.context_id == self.current_context:
            return 1.0
        if c.is_crystallized() and c.crucible_verified:
            return 1.0
        return 0.0

    def delta_R(self, z):
        if not self.centers:
            return 0.0
        K = self.kernel_batch(z)
        total = 0.0
        for i, c in enumerate(self.centers):
            g = self._read_gate(c)
            if g > 0 and K[i] > C.activation_thresh:
                total += g * c.v * K[i]
        return total

    def delta_k(self, z):
        if not self.centers or not self.enable_agency:
            return 0.0
        K = self.kernel_batch(z)
        total_w, sum_K = 0.0, 0.0
        for i, c in enumerate(self.centers):
            if not c.is_crystallized():
                continue
            g = self._read_gate(c)
            if g > 0 and K[i] > C.activation_thresh:
                total_w += g * c.w * K[i]
                sum_K += g * K[i]
        return total_w / sum_K if sum_K > 1e-6 else 0.0

    def R_eff(self, z):
        return float(np.clip(self.base_eval(z) + self.delta_R(z), 0.0, 1.0))

    def R_eff_batch(self, Z_flat):
        R_base = self.base_eval.batch(Z_flat)
        if not self.centers:
            return np.clip(R_base, 0.0, 1.0)
        Z_c = np.array([c.z for c in self.centers])
        sigmas = np.array([c.sigma for c in self.centers])
        vs = np.array([c.v for c in self.centers])
        gate = np.array([1.0 if (c.context_id == self.current_context
                                  or (c.is_crystallized() and c.crucible_verified))
                         else 0.0 for c in self.centers])
        Z_sq = np.sum(Z_flat**2, axis=1, keepdims=True)
        C_sq = np.sum(Z_c**2, axis=1, keepdims=True)
        sq_d = Z_sq + C_sq.T - 2.0 * (Z_flat @ Z_c.T)
        K = np.exp(-sq_d / (2.0 * sigmas[np.newaxis, :]**2))
        K[K < C.activation_thresh] = 0.0
        return np.clip(R_base + K @ (gate * vs), 0.0, 1.0)

    def k_eff(self, z):
        return max(C.k_min, C.k_0 + self.delta_k(z))

    def select_action(self, z_all, deterministic=False):
        R = np.array([self.R_eff(z) for z in z_all])
        if deterministic:
            return int(np.argmax(R)), R, np.full(C.k, C.k_0), 0.0
        k = np.array([self.k_eff(z) for z in z_all])
        logits = k * R
        logits -= logits.max()
        probs = np.exp(logits) / np.sum(np.exp(logits))
        chosen = int(np.random.choice(C.k, p=probs))
        entropy = float(-np.sum(probs * np.log(probs + 1e-12)))
        return chosen, R, k, entropy

    def _thermodynamic_shrink(self, center):
        if center.n_updates >= C.min_samples_shrink:
            dvar = center.D_var()
            calc = max(self._dynamic_sigma_floor,
                       self.sigma_base / (1.0 + C.alpha_shrink * dvar))
            center.sigma = min(center.sigma, calc)

    def _update_agency(self, center, k_weight):
        if center.is_crystallized():
            dv = center.D_var()
            tw = np.clip(C.w_max * (1.0 - dv / C.w_dvar_threshold),
                         -C.w_max, C.w_max)
            center.w += C.eta_k * k_weight * (tw - center.w)
            center.w = np.clip(center.w, -C.w_max, C.w_max)

    def update(self, z_chosen, D, x=None, j_chosen=None):
        self.global_step += 1
        K_all = self.kernel_batch(z_chosen) if self.centers else np.array([])

        # ── Pass 1: CRUCIBLE — cross-context crystals test residual
        if self.enable_crucible:
            for i, c in enumerate(self.centers):
                if c.is_crystallized() and c.context_id != self.current_context:
                    kw = float(K_all[i])
                    if kw >= C.activation_thresh:
                        juris_D = D * kw
                        c.v = np.clip(c.v + C.eta_cryst * juris_D, -C.v_max, C.v_max)
                        c.n_updates += 1
                        c.D_sum += juris_D
                        c.D_sq_sum += juris_D * juris_D
                        c.context_update_counts[self.current_context] += 1
                        c.D_history.append(D)  # raw D for verdict
                        c.D_step_history.append(self.global_step)
                        if x is not None:
                            c.xj_history.append((x.copy(), j_chosen))

        # ── Pass 2: LEARNING — same-context centers
        li = [i for i, c in enumerate(self.centers)
              if c.context_id == self.current_context]
        max_K = float(np.max(K_all[li])) if li else 0.0

        if max_K < C.creation_thresh:
            if len(self.centers) < C.capacity:
                juris_D = D * 1.0
                nc = MemoryCenter(
                    z=z_chosen.copy(),
                    v=np.clip(C.eta * juris_D, -C.v_max, C.v_max),
                    w=0.0, n_updates=1,
                    D_sum=juris_D, D_sq_sum=juris_D*juris_D,
                    mu_eff=C.mu_base,
                    context_id=self.current_context,
                    birth_step=self.global_step,
                    sigma=self.sigma_base)
                nc.context_update_counts[self.current_context] = 1
                nc.D_history.append(juris_D)
                nc.D_step_history.append(self.global_step)
                if x is not None:
                    nc.xj_history.append((x.copy(), j_chosen))
                self.centers.append(nc)
            return

        for i in li:
            c = self.centers[i]
            kw = float(K_all[i])
            if kw < C.activation_thresh:
                continue
            juris_D = D * kw
            effective_eta = C.eta_cryst if c.is_crystallized() else C.eta
            c.v = np.clip(c.v + effective_eta * juris_D, -C.v_max, C.v_max)
            c.n_updates += 1
            c.D_sum += juris_D
            c.D_sq_sum += juris_D * juris_D
            c.context_update_counts[self.current_context] += 1
            c.D_history.append(juris_D)
            c.D_step_history.append(self.global_step)
            if x is not None:
                c.xj_history.append((x.copy(), j_chosen))
            if self.enable_agency:
                self._update_agency(c, kw)
            self._thermodynamic_shrink(c)

    def end_epoch(self):
        for c in self.centers:
            c.v *= (1.0 - c.mu_eff)
            c.w *= (1.0 - c.mu_eff)

        for c in self.centers:
            if self.enable_crystallization:
                hist_len = len(c.D_history)
                cryst_grad = c.D_history[-50:] if hist_len > 0 else [0.0]
                cryst_mu = float(np.mean(cryst_grad))
                is_converged = abs(cryst_mu) < C.convergence_threshold

                if (not c.is_crystallized()
                        and c.n_updates >= C.cryst_n_min
                        and is_converged):
                    c.mu_eff = C.mu_cryst
                    c.was_ever_crystallized = True

                elif c.is_crystallized():
                    nc = c.n_cross_updates()
                    if nc >= C.n_cross_min:
                        crucible_grad = c.D_history[-C.n_cross_min:]
                        crucible_mu = float(np.mean(crucible_grad))
                        if (c.v * crucible_mu) < C.reversal_threshold:
                            c.dissolution_log.append({
                                'step': self.global_step,
                                'v': float(c.v),
                                'mu_D_recent': float(crucible_mu),
                                'product': float(c.v * crucible_mu),
                                'n_updates': c.n_updates,
                                'n_cross': nc,
                                'context': self.current_context,
                            })
                            c.mu_eff = C.mu_base
                            c.crucible_verified = False
                        else:
                            c.crucible_verified = True
            else:
                c.mu_eff = C.mu_base

        self._merge()

    def _merge(self):
        if len(self.centers) < 2:
            return
        merged = set()
        new = []
        for i in range(len(self.centers)):
            if i in merged:
                continue
            best = self.centers[i]
            for j in range(i+1, len(self.centers)):
                if j in merged:
                    continue
                if self.centers[i].context_id != self.centers[j].context_id:
                    continue
                dist = np.linalg.norm(self.centers[i].z - self.centers[j].z)
                ni = self.centers[i].sigma < self._needle_threshold
                nj = self.centers[j].sigma < self._needle_threshold
                if ni and nj:
                    th = self._merge_ratio * max(
                        self.centers[i].sigma, self.centers[j].sigma) * 1.5
                else:
                    th = self._merge_ratio * min(
                        self.centers[i].sigma, self.centers[j].sigma)
                if dist < th:
                    other = self.centers[j]
                    if other.n_updates > best.n_updates:
                        best, other = other, best
                    best.v = np.clip(best.v + other.v, -C.v_max, C.v_max)
                    best.w = np.clip(best.w + other.w, -C.w_max, C.w_max)
                    best.n_updates += other.n_updates
                    best.D_sum += other.D_sum
                    best.D_sq_sum += other.D_sq_sum
                    for ctx, cnt in other.context_update_counts.items():
                        best.context_update_counts[ctx] += cnt
                    best.D_history.extend(other.D_history)
                    best.D_step_history.extend(other.D_step_history)
                    best.xj_history.extend(other.xj_history)
                    best.sigma = min(best.sigma, other.sigma)
                    merged.add(j)
            new.append(best)
        if len(new) > C.capacity:
            cryst = [c for c in new if c.is_crystallized()]
            trans = sorted([c for c in new if not c.is_crystallized()],
                           key=lambda c: abs(c.v) * c.n_updates)
            keep = C.capacity - len(cryst)
            new = cryst + trans[-keep:] if keep > 0 else cryst[:C.capacity]
        self.centers = new

    def count_crystallized(self):
        return sum(1 for c in self.centers if c.is_crystallized())

    def count_verified(self):
        return sum(1 for c in self.centers
                   if c.is_crystallized() and c.crucible_verified)

    def sigma_stats(self):
        if not self.centers:
            return {}
        sigs = [c.sigma for c in self.centers]
        return {'mean': np.mean(sigs), 'min': np.min(sigs),
                'needles': sum(1 for s in sigs if s < self._needle_threshold)}

    def w_stats(self):
        if not self.centers:
            return {}
        ws = [c.w for c in self.centers]
        cws = [c.w for c in self.centers if c.is_crystallized()]
        return {'mean': float(np.mean(ws)), 'std': float(np.std(ws)),
                'frac_exploit': float(np.mean([w > 1.0 for w in ws])),
                'cryst_mean': float(np.mean(cws)) if cws else 0.0}

    def cryst_dvar_stats(self):
        dvars = [c.D_var() for c in self.centers if c.is_crystallized()]
        if not dvars:
            return 0.0, 0.0, 0.0
        return (float(np.median(dvars)),
                float(np.percentile(dvars, 25)),
                float(np.percentile(dvars, 75)))


# ── Engine state saving (new for paper) ──────────────────────────

def save_engine_state(ibf_agent, path):
    """Save FULL center data for post-hoc analysis.
    Includes D_history, xj_history, dissolution_log, context_update_counts —
    everything needed to reconstruct any metric without rerunning."""
    state = {
        'n_centers': len(ibf_agent.centers),
        'sigma_base': float(ibf_agent.sigma_base),
        'merge_thresh': float(ibf_agent.merge_thresh),
        'current_context': int(ibf_agent.current_context),
        'global_step': int(ibf_agent.global_step),
        'enable_crystallization': ibf_agent.enable_crystallization,
        'enable_crucible': ibf_agent.enable_crucible,
        'enable_agency': ibf_agent.enable_agency,
        'centers': [{
            'z': c.z.copy(),
            'v': float(c.v),
            'w': float(c.w),
            'sigma': float(c.sigma),
            'context_id': int(c.context_id),
            'is_crystallized': bool(c.is_crystallized()),
            'crucible_verified': bool(c.crucible_verified),
            'n_updates': int(c.n_updates),
            'D_var': float(c.D_var()),
            'D_sum': float(c.D_sum),
            'D_sq_sum': float(c.D_sq_sum),
            'was_ever_crystallized': bool(c.was_ever_crystallized),
            'mu_eff': float(c.mu_eff),
            'birth_step': int(c.birth_step),
            'context_update_counts': dict(c.context_update_counts),
            'D_history': list(c.D_history),
            'D_step_history': list(c.D_step_history),
            'xj_history': [(x.tolist(), int(j)) for x, j in c.xj_history],
            'dissolution_log': list(c.dissolution_log),
        } for c in ibf_agent.centers],
    }
    with open(path, 'wb') as f:
        pickle.dump(state, f)

In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — BASELINES (MLP, Replay MLP, Passive)                 ║
# ╚══════════════════════════════════════════════════════════════════╝

class MLPCorrection(nn.Module):
    def __init__(self, z_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.SiLU(),
                                 nn.Linear(hidden, 1), nn.Tanh())
        self.scale = 0.50
    def forward(self, z): return self.net(z).squeeze(-1) * self.scale

class MLPAgent:
    def __init__(self, encoder, base_eval, hidden=64, lr=None):
        self.encoder = encoder; self.base_eval = base_eval
        self.correction = MLPCorrection(C.d + C.k, hidden)
        self.optimizer = optim.SGD(self.correction.parameters(), lr=lr or C.mlp_lr)
    def R_eff(self, z):
        z_t = torch.tensor(z, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad(): corr = float(self.correction(z_t).item())
        return float(np.clip(self.base_eval(z) + corr, 0.0, 1.0))
    def R_eff_batch(self, Z_flat):
        R_base = self.base_eval.batch(Z_flat)
        z_t = torch.tensor(Z_flat, dtype=torch.float32)
        with torch.no_grad(): corr = self.correction(z_t).numpy()
        return np.clip(R_base + corr, 0.0, 1.0)
    def select_action(self, z_all, deterministic=False):
        R = np.array([self.R_eff(z) for z in z_all])
        if deterministic: return int(np.argmax(R)), R
        logits = C.k_0 * R; logits -= logits.max()
        probs = np.exp(logits) / np.sum(np.exp(logits))
        return int(np.random.choice(C.k, p=probs)), R
    def train_step(self, z_chosen, D):
        self.correction.train()
        z_t = torch.tensor(z_chosen, dtype=torch.float32).unsqueeze(0)
        corr = self.correction(z_t)
        loss = nn.MSELoss()(corr, (corr + D).detach())
        self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
        return float(loss.item())
    def param_count(self):
        return sum(p.numel() for p in self.correction.parameters())

class ReplayMLPAgent(MLPAgent):
    def __init__(self, *args, buffer_size=50, replay_per_step=1, **kwargs):
        super().__init__(*args, **kwargs)
        self.buffer = []; self.buffer_size = buffer_size
        self.replay_per_step = replay_per_step; self.steps_seen = 0
    def train_step_with_replay(self, z_chosen, D_current, R_imposed):
        self.steps_seen += 1; loss = self.train_step(z_chosen, D_current)
        for _ in range(self.replay_per_step):
            if self.buffer:
                idx = np.random.randint(len(self.buffer))
                zr, r_imp = self.buffer[idx]
                self.train_step(zr, r_imp - self.R_eff(zr))
        if len(self.buffer) < self.buffer_size:
            self.buffer.append((z_chosen.copy(), float(R_imposed)))
        else:
            j = np.random.randint(self.steps_seen)
            if j < self.buffer_size: self.buffer[j] = (z_chosen.copy(), float(R_imposed))
        return loss

class PassiveAgent:
    def __init__(self, encoder, base_eval):
        self.encoder = encoder; self.base_eval = base_eval
    def select_action(self, z_all, deterministic=True):
        R = np.array([self.base_eval(z) for z in z_all])
        return int(np.argmax(R)), R
    def R_eff_batch(self, Z_flat): return self.base_eval.batch(Z_flat)

print("Baselines ready.")

Baselines ready.


## Training & Evaluation

The next cells run the full experimental program:

1. **Cell 8**: Full IBF across 5 seeds with baselines alongside seed 42. Each seed trains 3 phases × 25 epochs × 1,000 positions. Engine states are saved after every phase for post-hoc analysis. Runtime: ~15 min per seed.

2. **Cells 8b/c/d**: Three ablations (No-Agency, No-Crystallization, No-Crucible), each across all 5 seeds. These isolate the causal contribution of each mechanism.

3. **Cell 9**: Aggregates results into Table 1 (§7.1) and runs a reproduction check.

### What to Look For

- **Phase A**: Accuracy rises from ~0.25 (chance) to ~0.92. Centers nucleate and crystallize.
- **Phase B**: Context switches. Phase A crystals are silenced, new centers nucleate. The Crucible tests sleeping crystals. Acc_A begins to decline as cross-context broadcast introduces interference.
- **Phase C**: A third context. Acc_C rises while Acc_A continues to decline — the thermodynamic cost of transfer in a maximally contradictory environment.
- **Ablation signatures**: No-Crucible gives BT_A ≈ 0 (safety is structural). No-Agency slightly improves retention (predicted regime effect in adversarial environments, §7.1).

In [8]:

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — TRAINING LOOP                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Full IBF (5 seeds) + baselines alongside seed 1.
# Engine states saved after each phase for every seed.

# ── Evaluation ───────────────────────────────────────────────────

def evaluate(agent, encoder, env, ctx):
    """Vectorized deterministic accuracy evaluation."""
    X = env.tests[ctx]
    N = len(X)
    Z_all = np.stack([encoder.encode_batch(X, np.full(N, j, dtype=int))
                      for j in range(C.k)], axis=1)
    Z_flat = Z_all.reshape(N * C.k, -1)
    R = agent.R_eff_batch(Z_flat).reshape(N, C.k)
    return float(np.mean(np.argmax(R, axis=1) == env.correct_actions_batch(X, ctx)))


# ── Crystal classification ───────────────────────────────────────

def classify_crystals(ibf, env):
    cats = {'invariant': [], 'context_specific': [], 'partial': [], 'other': []}
    spp = C.N_pool * C.E
    for ci, c in enumerate(ibf.centers):
        if c.birth_step > spp:
            continue
        if len(c.D_history) < 5 or len(c.xj_history) < 5:
            cats['other'].append(ci)
            continue
        n = min(len(c.D_history), len(c.xj_history))
        ds = np.array(c.D_history[:n])
        f1, fu, f2 = np.zeros(n), np.zeros(n), np.zeros(n)
        for t in range(n):
            x, j = c.xj_history[t]
            f1[t] = abs(x[0] * p_vec[j])
            fu[t] = abs(np.dot(env.u_A, x[2:4]) * r_vec[j])
            f2[t] = abs(x[1] * q_vec[j])
        w = np.abs(ds) + 1e-8
        s1 = np.average(f1, weights=w)
        su = np.average(fu, weights=w)
        s2 = np.average(f2, weights=w)
        total = s1 + su + s2 + 1e-8
        if s1/total > 0.45:
            cats['invariant'].append(ci)
        elif su/total > 0.45:
            cats['context_specific'].append(ci)
        elif s2/total > 0.35:
            cats['partial'].append(ci)
        else:
            cats['other'].append(ci)
    return cats


# ── BT_A helper ──────────────────────────────────────────────────

def _bt_a(lg, agent='ibf'):
    spp = C.N_pool * C.E
    idx_end_A = max((i for i, s in enumerate(lg['steps']) if s <= spp), default=0)
    return lg[agent]['A'][-1] - lg[agent]['A'][idx_end_A]


# ── Main experiment runner ───────────────────────────────────────

def run_experiment(seed, tag="full",
                   enable_crystallization=True,
                   enable_crucible=True,
                   enable_agency=True,
                   run_baselines=False,
                   quiet=False):
    """Run a single seed. Returns (log, ibf, cats, env, encoder, selectivity)."""
    global ACTION_EMB
    np.random.seed(seed)
    torch.manual_seed(seed)
    lp = (lambda *a, **kw: None) if quiet else print

    lp(f"\n{'='*70}")
    lp(f"  RRW v4.17 — {tag} (seed={seed})")
    lp(f"  η={C.eta} η_cryst={C.eta_cryst} μ_base={C.mu_base} μ_cryst={C.mu_cryst}")
    lp(f"  c_thresh={C.creation_thresh} a_thresh={C.activation_thresh} "
       f"rev_thresh={C.reversal_threshold}")
    lp(f"  w_max={C.w_max} n_cross={C.n_cross_min}")
    lp(f"{'='*70}")

    env = Environment(seed)
    encoder = RawEncoder(seed + 1)
    sigma_est, emb_scale = calibrate_action_embedding(encoder, seed, log_print=lp)
    base_eval = BaseEvaluator(seed + 2, encoder=encoder)
    eff_rank, sigma, merge_thresh = run_diagnostics(encoder, base_eval)

    ibf = IBFAgent(sigma, merge_thresh, encoder, base_eval,
                   enable_crystallization, enable_crucible, enable_agency)

    # Baselines only on first seed of full run
    mlp = MLPAgent(encoder, base_eval) if run_baselines else None
    replay_mlp = ReplayMLPAgent(encoder, base_eval, buffer_size=50) if run_baselines else None
    passive = PassiveAgent(encoder, base_eval) if run_baselines else None

    log = {
        'steps': [],
        'ibf': {'A':[],'B':[],'C':[]},
        'n_centers': [], 'n_crystallized': [], 'n_verified': [],
        'n_needles': [],
        'epoch_k_eff_mean': [], 'epoch_entropy': [],
        'epoch_w_mean': [],
        'emb_scale': emb_scale,
    }
    if run_baselines:
        for an in ['mlp', 'replay_mlp', 'passive']:
            log[an] = {'A':[],'B':[],'C':[]}

    total_step = 0
    t0 = time.time()

    for phase_name, ctx_id in [('A',0), ('B',1), ('C',2)]:
        lp(f"\n── Phase {phase_name} ──")
        ibf.set_context(ctx_id)
        pool = env.pools[phase_name]
        pt0 = time.time()

        for epoch in range(C.E):
            et0 = time.time()
            order = np.random.permutation(C.N_pool)
            ek, eH = [], []

            for idx in order:
                x = pool[idx]
                z_all = [encoder.encode(x, j) for j in range(C.k)]
                gt = env.correct_action_noisy(x, phase_name, epoch, idx)

                ch, Rv, kv, ent = ibf.select_action(z_all, deterministic=False)
                ek.extend(kv.tolist())
                eH.append(ent)
                Ri = 1.0 if ch == gt else 0.0
                Rc = ibf.R_eff(z_all[ch])
                ibf.update(z_all[ch], Ri - Rc, x=x, j_chosen=ch)

                if run_baselines:
                    mc, _ = mlp.select_action(z_all, deterministic=False)
                    mlp.train_step(z_all[mc], (1.0 if mc == gt else 0.0) - mlp.R_eff(z_all[mc]))

                    rc, _ = replay_mlp.select_action(z_all, deterministic=False)
                    Rir = 1.0 if rc == gt else 0.0
                    Rcr = replay_mlp.R_eff(z_all[rc])
                    replay_mlp.train_step_with_replay(z_all[rc], Rir - Rcr, Rir)

                total_step += 1
                if total_step % C.eval_every == 0:
                    log['steps'].append(total_step)
                    saved_ctx = ibf.current_context
                    for ci, ctx in enumerate('ABC'):
                        ibf.set_context(ci)
                        log['ibf'][ctx].append(evaluate(ibf, encoder, env, ctx))
                        if run_baselines:
                            log['mlp'][ctx].append(evaluate(mlp, encoder, env, ctx))
                            log['replay_mlp'][ctx].append(evaluate(replay_mlp, encoder, env, ctx))
                            log['passive'][ctx].append(evaluate(passive, encoder, env, ctx))
                    ibf.set_context(saved_ctx)
                    log['n_centers'].append(len(ibf.centers))
                    log['n_crystallized'].append(ibf.count_crystallized())
                    log['n_verified'].append(ibf.count_verified())
                    ss = ibf.sigma_stats()
                    log['n_needles'].append(ss.get('needles', 0))

            ibf.end_epoch()
            ws = ibf.w_stats()
            log['epoch_k_eff_mean'].append(float(np.mean(ek)) if ek else C.k_0)
            log['epoch_entropy'].append(float(np.mean(eH)) if eH else 0.0)
            log['epoch_w_mean'].append(ws.get('mean', 0.0))

            nc = len(ibf.centers)
            nx = ibf.count_crystallized()
            nv = ibf.count_verified()
            acc_str = ""
            if log['ibf']['A']:
                acc_str = (f"  IBF [{log['ibf']['A'][-1]:.3f}/"
                           f"{log['ibf']['B'][-1]:.3f}/{log['ibf']['C'][-1]:.3f}]")
            lp(f"  Ep {epoch+1:>2d}/{C.E}  {nc} ctr ({nx} cryst, {nv} vrf)  "
               f"k={log['epoch_k_eff_mean'][-1]:.2f} H={log['epoch_entropy'][-1]:.3f}  "
               f"{time.time()-et0:.1f}s{acc_str}")

        # ── Save engine state after each phase ───────────────────
        state_path = os.path.join(
            OUTPUT_DIR, f"engine_{tag}_s{seed}_phase{phase_name}.pkl")
        save_engine_state(ibf, state_path)
        lp(f"  Saved engine state: {state_path}")

        pe = time.time() - pt0
        ws = ibf.w_stats()
        md, p25, p75 = ibf.cryst_dvar_stats()
        lp(f"  ── End {phase_name} ({pe:.0f}s): {len(ibf.centers)} ctr "
           f"({ibf.count_crystallized()} cryst, {ibf.count_verified()} verified) ──")
        lp(f"  Agency: w_mean={ws.get('mean',0):.3f} cryst_w={ws.get('cryst_mean',0):.3f}")
        lp(f"  Cryst D_var: med={md:.4f} (p25={p25:.4f}, p75={p75:.4f})")
        if log['ibf']['A']:
            lp(f"         ibf Acc: A={log['ibf']['A'][-1]:.3f} "
               f"B={log['ibf']['B'][-1]:.3f} C={log['ibf']['C'][-1]:.3f}")
            if run_baselines:
                for an in ['mlp', 'replay_mlp', 'passive']:
                    lp(f"  {an:>12s} Acc: A={log[an]['A'][-1]:.3f} "
                       f"B={log[an]['B'][-1]:.3f} C={log[an]['C'][-1]:.3f}")

    lp(f"\n  Total: {time.time()-t0:.1f}s")

    # ── Crystal classification & selectivity ─────────────────────
    cats = classify_crystals(ibf, env)
    for cn, idx in cats.items():
        lp(f"  Crystal '{cn}': {len(idx)} centers")

    def _melt_stats(cat_name):
        indices = cats.get(cat_name, [])
        ever = [ci for ci in indices
                if ci < len(ibf.centers) and ibf.centers[ci].was_ever_crystallized]
        melted = sum(1 for ci in ever if ibf.centers[ci].dissolution_log)
        return len(indices), len(ever), melted

    _, inv_ever, inv_melted = _melt_stats('invariant')
    _, ctx_ever, ctx_melted = _melt_stats('context_specific')
    inv_mr = inv_melted / max(inv_ever, 1)
    ctx_mr = ctx_melted / max(ctx_ever, 1)

    # ── Selectivity report ───────────────────────────────────────
    lp(f"\n  ── Selectivity Report ──")
    lp(f"  Invariant:        {inv_ever} ever cryst, {inv_melted} melted ({inv_mr:.0%})")
    lp(f"  Context-specific: {ctx_ever} ever cryst, {ctx_melted} melted ({ctx_mr:.0%})")
    if inv_mr == 0:
        lp(f"  Selectivity: PERFECT invariant survival (ctx_melt={ctx_mr:.0%})")
    else:
        lp(f"  Selectivity ratio: {ctx_mr / inv_mr:.1f}x")

    # ── Dissolution events ───────────────────────────────────────
    cats_lookup = {ci: cat for cat, indices in cats.items() for ci in indices}
    all_diss = []
    for ci, c in enumerate(ibf.centers):
        for evt in c.dissolution_log:
            all_diss.append({**evt, 'center_idx': ci,
                             'category': cats_lookup.get(ci, 'unknown')})
    if all_diss:
        lp(f"\n  ── Dissolution Events ({len(all_diss)}) ──")
        for ev in sorted(all_diss, key=lambda e: e['step']):
            lp(f"    step={ev['step']} cat={ev['category']:<18s} "
               f"v={ev['v']:.4f} mu_D={ev['mu_D_recent']:.4f} "
               f"prod={ev['product']:.4f} n_cross={ev['n_cross']}")

    # ── Crystal stats per seed ───────────────────────────────────
    lp(f"\n── Crystal Stats (seed={seed}) ──")
    for cat, indices in cats.items():
        surv = sum(1 for i in indices if i < len(ibf.centers)
                   and ibf.centers[i].is_crystallized())
        ever = sum(1 for i in indices if i < len(ibf.centers)
                   and ibf.centers[i].was_ever_crystallized)
        vrfd = sum(1 for i in indices if i < len(ibf.centers)
                   and ibf.centers[i].crucible_verified)
        lp(f"  {cat}: {len(indices)} total, {ever} ever cryst, "
           f"{surv} surviving, {vrfd} verified")

    selectivity = {
        'inv_ever': inv_ever, 'inv_melted': inv_melted, 'inv_mr': inv_mr,
        'ctx_ever': ctx_ever, 'ctx_melted': ctx_melted, 'ctx_mr': ctx_mr,
    }

    return log, ibf, cats, env, encoder, selectivity


# ══════════════════════════════════════════════════════════════════
# RUN: Full IBF (5 seeds) + baselines on seed 1
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("  FULL IBF — 5 SEEDS")
print("="*70)

all_results = {}
multi_seed_logs = {'full': []}

for si, seed in enumerate(SEEDS):
    log, ibf, cats, env, enc, sel = run_experiment(
        seed=seed, tag=f"full_s{seed}",
        run_baselines=True)
    all_results[f"full_s{seed}"] = (log, sel)
    multi_seed_logs['full'].append(log)

# ── κ_rrw computation (run once) ─────────────────────────────────
print("\n" + "="*70)
print("  κ_rrw COMPUTATION")
print("="*70)
test_enc = RawEncoder(SEEDS[0] + 1)
calibrate_action_embedding(test_enc, SEEDS[0], log_print=lambda *a: None)
kappa_result = compute_kappa(test_enc, SEEDS[0])

# ── Multi-seed summary ───────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  MULTI-SEED SUMMARY (Full IBF)")
print(f"{'='*70}")
for an in ['ibf']:
    aA = [lg[an]['A'][-1] for lg in multi_seed_logs['full']]
    bt = [_bt_a(lg, an) for lg in multi_seed_logs['full']]
    aC = [lg[an]['C'][-1] for lg in multi_seed_logs['full']]
    print(f"  {an:>12s}  Acc_A={np.mean(aA):.3f}±{np.std(aA):.3f}  "
          f"BT_A={np.mean(bt):+.3f}±{np.std(bt):.3f}  "
          f"Acc_C={np.mean(aC):.3f}±{np.std(aC):.3f}")

# Baselines from seed 1 only
log_s1 = multi_seed_logs['full'][0]
if 'mlp' in log_s1:
    for an in ['mlp', 'replay_mlp', 'passive']:
        print(f"  {an:>12s}  Acc_A={log_s1[an]['A'][-1]:.3f}  "
              f"BT_A={_bt_a(log_s1, an):+.3f}  "
              f"Acc_C={log_s1[an]['C'][-1]:.3f}  (seed 42 only)")


  FULL IBF — 5 SEEDS

  RRW v4.17 — full_s42 (seed=42)
  η=0.1 η_cryst=0.005 μ_base=0.06 μ_cryst=0.001
  c_thresh=0.3 a_thresh=0.18 rev_thresh=-0.125
  w_max=5.0 n_cross=8
  [Sep iter 1] sep=1.4142, σ=0.8933, ratio=1.58
  Rescaled ACTION_EMB 2.53x (cumulative: 2.53x)
  [Sep iter 2] sep=3.5732, σ=0.8933, ratio=4.00
  Action separation OK (ratio=4.00)
  Base evaluator spread: 0.0368 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4

── Phase A ──
  Ep  1/25  201 ctr (42 cryst, 0 vrf)  k=5.00 H=1.228  5.9s  IBF [0.854/0.243/0.246]
  Ep  2/25  249 ctr (88 cryst, 0 vrf)  k=5.66 H=0.822  6.8s  IBF [0.889/0.243/0.246]
  Ep  3/25  276 ctr (114 cryst, 0 vrf)  k=6.13 H=0.666  8.1s  IBF [0.904/0.243/0.246]
  Ep  4/25  296 ctr (126 cryst, 0 vrf)  k=6.51 H=0.560  9.0s  IBF [0.913/0.243/0.246]
  Ep  5/25  300 ctr (136 cryst, 0 vrf)  k=6.71 H=0.498  8.9s  IBF [0.916/0.243/0.246]
  Ep  6/25  302 ctr (143 cryst, 0 vrf)  k=6.88 H=0.468  9.0s  IBF [0.917/0.243/0.246]
  Ep  7/25  307 ctr (152 cry

In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 8b/c/d — ABLATIONS (No-Agency, No-Cryst, No-Crucible)    ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── No-Agency ────────────────────────────────────────────────────
print(f"\n{'='*70}\n  ABLATION: No-Agency (5 seeds)\n{'='*70}")
multi_seed_logs['no_agency'] = []
for seed in SEEDS:
    saved_eta_k, saved_w_max = C.eta_k, C.w_max
    try:
        C.eta_k = 0.0; C.w_max = 0.0
        log_a, _, _, _, _, _ = run_experiment(seed=seed, tag=f"no_agency_s{seed}",
                                              enable_agency=False, quiet=True)
    finally:
        C.eta_k, C.w_max = saved_eta_k, saved_w_max
    multi_seed_logs['no_agency'].append(log_a)

# ── No-Crystallization ───────────────────────────────────────────
print(f"\n{'='*70}\n  ABLATION: No-Crystallization (5 seeds)\n{'='*70}")
multi_seed_logs['no_cryst'] = []
for seed in SEEDS:
    log_a, _, _, _, _, _ = run_experiment(seed=seed, tag=f"no_cryst_s{seed}",
                                          enable_crystallization=False, quiet=True)
    multi_seed_logs['no_cryst'].append(log_a)

# ── No-Crucible ──────────────────────────────────────────────────
print(f"\n{'='*70}\n  ABLATION: No-Crucible (5 seeds)\n{'='*70}")
multi_seed_logs['no_crucible'] = []
for seed in SEEDS:
    log_a, _, _, _, _, _ = run_experiment(seed=seed, tag=f"no_crucible_s{seed}",
                                          enable_crucible=False, quiet=True)
    multi_seed_logs['no_crucible'].append(log_a)

print("\nAll ablations complete.")


  ABLATION: No-Agency (5 seeds)
  Base evaluator spread: 0.0368 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0283 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0356 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0483 (init variance=0.03)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0253 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4

  ABLATION: No-Crystallization (5 seeds)
  Base evaluator spread: 0.0368 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0283 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0356 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0483 (init variance=0.03)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0253 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4

  ABLATION: No-Crucible 

In [10]:

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — EVALUATION & METRICS                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

print(f"\n{'='*100}")
print(f"  TABLE 1 — RRW v4.17 MULTI-SEED RESULTS")
print(f"{'='*100}")
print(f"  {'Condition':<20s} {'Acc_A(end)':<16s} {'BT_A':<16s} {'Acc_C(end)':<16s}")
print(f"  {'-'*65}")

for cond in ['full', 'no_agency', 'no_crucible', 'no_cryst']:
    logs = multi_seed_logs[cond]
    aA = [lg['ibf']['A'][-1] for lg in logs]
    bt = [_bt_a(lg) for lg in logs]
    aC = [lg['ibf']['C'][-1] for lg in logs]
    print(f"  {cond:<20s} {np.mean(aA):.3f} ± {np.std(aA):.3f}   "
          f"{np.mean(bt):+.3f} ± {np.std(bt):.3f}   "
          f"{np.mean(aC):.3f} ± {np.std(aC):.3f}")

# Baselines from seed 1
if 'mlp' in multi_seed_logs['full'][0]:
    for an in ['mlp', 'replay_mlp']:
        logs = multi_seed_logs['full']
        # Only have baselines for seed 1; report that
        lg = logs[0]
        aA_v = lg[an]['A'][-1]
        bt_v = _bt_a(lg, an)
        aC_v = lg[an]['C'][-1]
        print(f"  {an:<20s} {aA_v:.3f}           "
              f"{bt_v:+.3f}            "
              f"{aC_v:.3f}            (seed 42)")

# ── Expected results check ───────────────────────────────────────
print(f"\n── Reproduction Check ──")
expected = {
    'full':        {'Acc_A': 0.684, 'BT_A': -0.234, 'Acc_C': 0.913},
    'no_agency':   {'Acc_A': 0.717, 'BT_A': -0.201, 'Acc_C': 0.911},
    'no_crucible': {'Acc_A': 0.909, 'BT_A': -0.009, 'Acc_C': 0.912},
    'no_cryst':    {'Acc_A': 0.755, 'BT_A': -0.160, 'Acc_C': 0.911},
}
all_ok = True
for cond, exp in expected.items():
    logs = multi_seed_logs[cond]
    aA = np.mean([lg['ibf']['A'][-1] for lg in logs])
    bt = np.mean([_bt_a(lg) for lg in logs])
    aC = np.mean([lg['ibf']['C'][-1] for lg in logs])
    aA_std = np.std([lg['ibf']['A'][-1] for lg in logs])
    bt_std = np.std([_bt_a(lg) for lg in logs])
    aC_std = np.std([lg['ibf']['C'][-1] for lg in logs])

    checks = []
    for name, got, exp_v, std in [('Acc_A', aA, exp['Acc_A'], aA_std),
                                    ('BT_A', bt, exp['BT_A'], bt_std),
                                    ('Acc_C', aC, exp['Acc_C'], aC_std)]:
        tol = max(std, 0.01)  # at least 1 std or 0.01
        ok = abs(got - exp_v) <= tol
        checks.append(('✓' if ok else '✗', name, got, exp_v))
        if not ok:
            all_ok = False

    status = " ".join(f"{s}{n}={g:.3f}(exp={e:.3f})" for s,n,g,e in checks)
    print(f"  {cond:<15s}: {status}")

print(f"\n  {'ALL REPRODUCED WITHIN TOLERANCE' if all_ok else 'WARNING: SOME VALUES OUTSIDE TOLERANCE'}")



  TABLE 1 — RRW v4.17 MULTI-SEED RESULTS
  Condition            Acc_A(end)       BT_A             Acc_C(end)      
  -----------------------------------------------------------------
  full                 0.684 ± 0.051   -0.234 ± 0.052   0.913 ± 0.008
  no_agency            0.721 ± 0.047   -0.198 ± 0.052   0.918 ± 0.006
  no_crucible          0.911 ± 0.007   -0.005 ± 0.003   0.921 ± 0.003
  no_cryst             0.767 ± 0.103   -0.152 ± 0.108   0.920 ± 0.008
  mlp                  0.442           -0.522            0.489            (seed 42)
  replay_mlp           0.536           -0.410            0.816            (seed 42)

── Reproduction Check ──
  full           : ✓Acc_A=0.684(exp=0.684) ✓BT_A=-0.234(exp=-0.234) ✓Acc_C=0.913(exp=0.913)
  no_agency      : ✓Acc_A=0.721(exp=0.717) ✓BT_A=-0.198(exp=-0.201) ✓Acc_C=0.918(exp=0.911)
  no_crucible    : ✓Acc_A=0.911(exp=0.909) ✓BT_A=-0.005(exp=-0.009) ✓Acc_C=0.921(exp=0.912)
  no_cryst       : ✓Acc_A=0.767(exp=0.755) ✓BT_A=-0.152(exp=-0.160

## Post-Training Analysis

### σ_eval Sweep (Cell 10)

After training at the geometrically calibrated bandwidth σ* = 0.89, we evaluate across a range of σ_eval values without retraining. In RRW, the paper (§7.1) reports the peak at **scale 1.1**, essentially flat relative to training scale. In a maximally contradictory environment, wider readout aggregates contradictory corrections, so the benefit remains modest — unlike chess, where the Crucible makes wider readout safe.

### κ Report

Cell 10 also reports the geometric scaling ratio κ = σ*/√d_eff for the three-domain comparison table (§5.1). RRW: κ = 0.45.

In [13]:

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — σ_eval SWEEP & κ COMPUTATION                        ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Run σ_eval sweep on all 5 seed final engine states.
# Identify the peak σ_eval for BT_A.

print(f"\n{'='*70}")
print(f"  σ_eval SWEEP")
print(f"{'='*70}")

sweep_results = {scale: {'Acc_A': [], 'Acc_B': [], 'Acc_C': [], 'BT_A': []}
                 for scale in SIGMA_EVAL_SCALES}

for seed in SEEDS:
    # Load engine state from Phase C
    state_path = os.path.join(OUTPUT_DIR, f"engine_full_s{seed}_s{seed}_phaseC.pkl")
    if not os.path.exists(state_path):
        print(f"  WARNING: Missing {state_path}, skipping seed {seed}")
        continue

    with open(state_path, 'rb') as f:
        state = pickle.load(f)

    # Also need phase A end accuracy for BT_A
    state_path_A = os.path.join(OUTPUT_DIR, f"engine_full_s{seed}_phaseA.pkl")

    # Reconstruct a minimal IBF for evaluation
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = Environment(seed)
    encoder = RawEncoder(seed + 1)
    calibrate_action_embedding(encoder, seed, log_print=lambda *a: None)
    base_eval = BaseEvaluator(seed + 2, encoder=encoder)
    _, sigma_cal, merge_t = run_diagnostics(encoder, base_eval)

    # Get Acc_A at end of Phase A from the full log
    log_full = multi_seed_logs['full'][SEEDS.index(seed)]
    spp = C.N_pool * C.E
    idx_end_A = max((i for i, s in enumerate(log_full['steps']) if s <= spp), default=0)
    acc_A_end_phaseA = log_full['ibf']['A'][idx_end_A]

    for scale in SIGMA_EVAL_SCALES:
        # Build IBF agent from saved state with modified sigmas
        ibf_eval = IBFAgent(sigma_cal, merge_t, encoder, base_eval)
        for cd in state['centers']:
            mc = MemoryCenter(
                z=cd['z'].copy(),
                v=cd['v'], w=cd['w'],
                sigma=cd['sigma'] * scale,  # Apply scale relative to trained σ
                context_id=cd['context_id'],
                mu_eff=cd['mu_eff'],
                n_updates=cd['n_updates'],
            )
            mc.crucible_verified = cd['crucible_verified']
            mc.was_ever_crystallized = cd.get('was_ever_crystallized',
                                              cd['is_crystallized'])
            ibf_eval.centers.append(mc)
        ibf_eval.global_step = state['global_step']

        # Evaluate all three contexts
        accs = {}
        for ci, ctx in enumerate('ABC'):
            ibf_eval.set_context(ci)
            accs[ctx] = evaluate(ibf_eval, encoder, env, ctx)

        bt_a = accs['A'] - acc_A_end_phaseA

        sweep_results[scale]['Acc_A'].append(accs['A'])
        sweep_results[scale]['Acc_B'].append(accs['B'])
        sweep_results[scale]['Acc_C'].append(accs['C'])
        sweep_results[scale]['BT_A'].append(bt_a)

# ── Print sweep table ────────────────────────────────────────────
print(f"\n  {'σ_scale':<10s} {'Acc_A':<16s} {'BT_A':<16s} {'Acc_C':<16s}")
print(f"  {'-'*58}")

best_scale, best_bt = None, -999
for scale in SIGMA_EVAL_SCALES:
    sr = sweep_results[scale]
    if not sr['Acc_A']:
        continue
    mA = np.mean(sr['Acc_A'])
    mBT = np.mean(sr['BT_A'])
    mC = np.mean(sr['Acc_C'])
    sA = np.std(sr['Acc_A'])
    sBT = np.std(sr['BT_A'])
    sC = np.std(sr['Acc_C'])
    marker = ""
    if mBT > best_bt:
        best_bt = mBT
        best_scale = scale
        marker = " ←"
    print(f"  {scale:<10.1f} {mA:.3f}±{sA:.3f}     {mBT:+.3f}±{sBT:.3f}     "
          f"{mC:.3f}±{sC:.3f}{marker}")

if best_scale is not None:
    print(f"\n  σ_eval peak: scale={best_scale:.1f} "
          f"(BT_A={best_bt:+.3f})")

# ── κ report ─────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  THREE-DOMAIN TABLE (RRW row)")
print(f"{'='*70}")
print(f"  Domain    z_dim  d_eff   σ_train  σ_eval_peak  κ        BT_A")
print(f"  {'-'*68}")
print(f"  RRW       8      {kappa_result['d_eff']:.1f}    "
      f"{kappa_result['sigma']:.4f}  "
      f"{best_scale if best_scale else '?':<12}  "
      f"{kappa_result['kappa']:.4f}   "
      f"{np.mean([_bt_a(lg) for lg in multi_seed_logs['full']]):+.3f}")
print(f"  Chess     12     ?       1.3      ?(≈1.0)      ?        +14.0")
print(f"  CIFAR     68     ?       ?        ?            ?        ?")

# ── Ablation cascade table ───────────────────────────────────────
print(f"\n  Ablation Cascade:")
print(f"  {'':15s} {'Full IBF':>10s} {'No-Agency':>10s} "
      f"{'No-Cryst':>10s} {'No-Crucible':>12s} {'MLP':>8s} {'Replay':>8s}")
for cond, label in [('full','RRW BT_A')]:
    vals = []
    for c in ['full', 'no_agency', 'no_cryst', 'no_crucible']:
        bt = np.mean([_bt_a(lg) for lg in multi_seed_logs[c]])
        vals.append(f"{bt:+.3f}")
    # MLP and Replay from seed 1
    lg1 = multi_seed_logs['full'][0]
    if 'mlp' in lg1:
        vals.append(f"{_bt_a(lg1,'mlp'):+.3f}")
        vals.append(f"{_bt_a(lg1,'replay_mlp'):+.3f}")
    else:
        vals.extend(['?', '?'])
    print(f"  {label:<15s} {'  '.join(vals)}")

# ── Save all results to JSON ─────────────────────────────────────
json_out = {
    'version': 'v4.17_clean',
    'seeds': SEEDS,
    'config': {k: v for k, v in vars(C).items() if not k.startswith('_')},
    'kappa': kappa_result,
    'sigma_eval_sweep': {
        str(scale): {
            'Acc_A_mean': float(np.mean(sweep_results[scale]['Acc_A'])),
            'Acc_A_std': float(np.std(sweep_results[scale]['Acc_A'])),
            'BT_A_mean': float(np.mean(sweep_results[scale]['BT_A'])),
            'BT_A_std': float(np.std(sweep_results[scale]['BT_A'])),
            'Acc_C_mean': float(np.mean(sweep_results[scale]['Acc_C'])),
            'Acc_C_std': float(np.std(sweep_results[scale]['Acc_C'])),
        }
        for scale in SIGMA_EVAL_SCALES if sweep_results[scale]['Acc_A']
    },
    'sigma_eval_peak': float(best_scale) if best_scale else None,
    'multi_seed_summary': {},
    'ablations': {},
    'selectivity': {
        'inv_mr_mean': float(np.mean([all_results[f"full_s{s}"][1]['inv_mr'] for s in SEEDS])),
        'ctx_mr_mean': float(np.mean([all_results[f"full_s{s}"][1]['ctx_mr'] for s in SEEDS])),
    },
}

for cond in ['full', 'no_agency', 'no_crucible', 'no_cryst']:
    logs = multi_seed_logs[cond]
    aA = [lg['ibf']['A'][-1] for lg in logs]
    bt = [_bt_a(lg) for lg in logs]
    aC = [lg['ibf']['C'][-1] for lg in logs]
    json_out['multi_seed_summary' if cond == 'full' else 'ablations'][
        cond if cond != 'full' else 'ibf'] = {
        'accA_mean': float(np.mean(aA)), 'accA_std': float(np.std(aA)),
        'btA_mean': float(np.mean(bt)), 'btA_std': float(np.std(bt)),
        'accC_mean': float(np.mean(aC)), 'accC_std': float(np.std(aC)),
    }

# Baselines from seed 1
if 'mlp' in multi_seed_logs['full'][0]:
    for an in ['mlp', 'replay_mlp', 'passive']:
        lg = multi_seed_logs['full'][0]
        json_out['multi_seed_summary'][an] = {
            'accA': float(lg[an]['A'][-1]),
            'btA': float(_bt_a(lg, an)),
            'accC': float(lg[an]['C'][-1]),
            'note': 'seed 42 only',
        }

json_path = os.path.join(OUTPUT_DIR, 'rrw_v417_clean_results.json')
with open(json_path, 'w') as f:
    json.dump(json_out, f, indent=2)
print(f"\n  Saved: {json_path}")

# ── Final status ─────────────────────────────────────────────────
inv_mrs = [all_results[f"full_s{s}"][1]['inv_mr'] for s in SEEDS]
ctx_mrs = [all_results[f"full_s{s}"][1]['ctx_mr'] for s in SEEDS]
n_perfect = sum(1 for m in inv_mrs if m == 0)

print(f"\n{'='*70}")
print(f"  RRW v4.17 PAPER-GRADE RUN — COMPLETE")
print(f"  Selectivity: inv={np.mean(inv_mrs):.0%} ctx={np.mean(ctx_mrs):.0%} "
      f"perfect_inv={n_perfect}/{len(SEEDS)}")
print(f"  κ_rrw = {kappa_result['kappa']:.4f}")
print(f"  σ_eval peak = {best_scale}")
print(f"  Engine states saved for {len(SEEDS)} seeds × 4 conditions × 3 phases")
print(f"{'='*70}")


  σ_eval SWEEP
  Base evaluator spread: 0.0368 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0283 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0356 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0483 (init variance=0.03)
  σ=0.8933, merge=0.8933, eff_rank=4
  Base evaluator spread: 0.0253 (init variance=0.02)
  σ=0.8933, merge=0.8933, eff_rank=4

  σ_scale    Acc_A            BT_A             Acc_C           
  ----------------------------------------------------------
  0.7        0.599±0.043     -0.319±0.043     0.762±0.013 ←
  0.8        0.648±0.048     -0.270±0.048     0.835±0.009 ←
  0.9        0.674±0.054     -0.244±0.055     0.883±0.008 ←
  1.0        0.684±0.052     -0.235±0.052     0.913±0.008 ←
  1.1        0.690±0.048     -0.229±0.049     0.931±0.006 ←
  1.2        0.685±0.050     -0.233±0.052     0.939±0.006
  1.3        0.677±0.045     -0.241±0.046     0.93